# 02: Cleaning and open-coding selection

**Exclusion decisions**

- Exclude outlets listed in `config/global_source_blacklist.csv` (publisher headquartered outside North America).
- Exclude records not in English, missing a usable title/opening, or without **Zohran** or **Mamdani** in the title, description, or snippet. This deliberately uses a conservative relevance threshold.
- **Keep** discussion-forum threads, sponsored posts, video news reports, and opinion/editorial articles when they meet the other criteria.
- Exclude duplicate UUIDs or article URLs (keep first occurrence). Different publishers covering the same story remain distinct articles.
- Exclude `d7c6ac96-a623-4845-bbc7-4a69cc6b3a6e` (*Donald Trump Is Writing Fan Fiction About the L.A. Protests*): its headline and description are unrelated to Mamdani, but the provider included a mismatched snippet.

All excluded records and their reasons are saved in `data/cleaned/<annotator>_excluded.csv`. The 50-article sample is uniform random (fixed seed) and contains blank open-coding columns.

In [1]:
from pathlib import Path
from urllib.parse import urlsplit
from dotenv import load_dotenv
import os
import re
import pandas as pd

In [2]:
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/csv_data").is_dir()), None)
if root is None:
    raise FileNotFoundError("Run this notebook from the project or notebooks/ folder.")
load_dotenv(root / ".env")

annotator = os.getenv("ANNOTATOR", "").strip().lower()
start_date = os.getenv("START_DATE", "").strip()
end_date = os.getenv("END_DATE", "").strip()
if not re.fullmatch(r"[a-z0-9_-]+", annotator):
    raise ValueError("Add ANNOTATOR=yoon to your .env file (letters, digits, _ or - only).")
for value, name in [(start_date, "START_DATE"), (end_date, "END_DATE")]:
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        raise ValueError(f"Set {name}=YYYY-MM-DD in .env")

raw_file = root / "data/raw/csv_data" / f"{start_date}__{end_date}.csv"
blacklist_file = root / "config/global_source_blacklist.csv"
if not raw_file.is_file() or not blacklist_file.is_file():
    raise FileNotFoundError(f"Missing file: {raw_file if not raw_file.is_file() else blacklist_file}")

articles = pd.read_csv(raw_file, dtype=str, keep_default_na=False)
blacklist = pd.read_csv(blacklist_file, dtype=str, keep_default_na=False)
required = {"uuid", "title", "description", "snippet", "url", "language", "source", "published_at"}
if missing := required - set(articles.columns):
    raise ValueError(f"Missing article columns: {sorted(missing)}")
if "source" not in blacklist:
    raise ValueError("Blacklist must contain a 'source' column")

def domain(value):
    value = str(value).strip().lower()
    hostname = urlsplit(value).hostname if "://" in value else value
    return (hostname or "").removeprefix("www.").split(":")[0]

blocked = {domain(value) for value in blacklist["source"] if str(value).strip()}
articles["source_domain"] = articles["source"].map(domain)
articles["url_key"] = articles["url"].map(lambda value: value.strip().split("?")[0].rstrip("/").lower())

# For coding use the title and opening, not arbitrary keyword metadata
def get_opening(row):
    description = row["description"].strip()
    snippet = row["snippet"].strip()
    if description.casefold() not in {"", "the nation magazine", "n/a", "none"}:
        return re.sub(r"\s+", " ", description).strip()
    snippet = re.sub(r"Getting your Trinity Audio player ready\.\.\.|NEW You can now listen to Fox News articles!|Your browser does not support the video tag\.", "", snippet, flags=re.I)
    return re.sub(r"\s+", " ", snippet).strip()

articles["opening"] = articles.apply(get_opening, axis=1)
print(f"Loaded {len(articles)} articles from {raw_file.name} for {annotator}.")

Loaded 210 articles from 2025-01-01__2025-06-24.csv for yoon.


In [3]:
# Apply all decisions in one run; keep a reason for every exclusion
subject = re.compile(r"\b(?:zohran|mamdani)\b", flags=re.I)
known_bad = {"d7c6ac96-a623-4845-bbc7-4a69cc6b3a6e": "mismatched_api_snippet"}
seen_ids, seen_urls = set(), set()

reasons = []
for _, article in articles.iterrows():
    issues = []
    source = article["source_domain"]
    if any(source == blocked_domain or source.endswith("." + blocked_domain) for blocked_domain in blocked):
        issues.append("non_north_american_publisher")
    if article["language"].strip().lower() != "en":
        issues.append("non_english")
    excerpt = " ".join([article["title"], article["description"], article["snippet"]])
    if not subject.search(excerpt):
        issues.append("no_subject_mention_in_available_text")
    if not article["title"].strip() or len(article["opening"]) < 25:
        issues.append("missing_title_or_usable_opening")
    if article["uuid"] in known_bad:
        issues.append(known_bad[article["uuid"]])
    if not article["uuid"].strip() or article["uuid"] in seen_ids:
        issues.append("missing_or_duplicate_uuid")
    if not article["url_key"] or article["url_key"] in seen_urls:
        issues.append("missing_or_duplicate_url")
    seen_ids.add(article["uuid"])
    seen_urls.add(article["url_key"])
    reasons.append("; ".join(issues))

articles["exclusion_reason"] = reasons
excluded = articles.loc[articles["exclusion_reason"].ne("")].copy()
cleaned = articles.loc[articles["exclusion_reason"].eq("")].copy()
cleaned["collector"] = annotator

output_dir = root / "data/cleaned"
output_dir.mkdir(parents=True, exist_ok=True)
cleaned_path = output_dir / f"{annotator}_cleaned.csv"
excluded_path = output_dir / f"{annotator}_excluded.csv"

cleaned.drop(columns=["url_key", "exclusion_reason"]).to_csv(cleaned_path, index=False, encoding="utf-8-sig")
excluded[["uuid", "source", "title", "url", "exclusion_reason"]].to_csv(excluded_path, index=False, encoding="utf-8-sig")

print(f"Raw: {len(articles)} | Excluded: {len(excluded)} | Cleaned: {len(cleaned)}")
if not excluded.empty:
    print("\nExclusion reasons (one article may have multiple):")
    print(excluded["exclusion_reason"].str.split("; ").explode().value_counts().to_string())
print(f"\nSaved: {cleaned_path}")
print(f"Saved: {excluded_path}")

Raw: 210 | Excluded: 91 | Cleaned: 119

Exclusion reasons (one article may have multiple):
exclusion_reason
no_subject_mention_in_available_text    69
non_north_american_publisher            26
mismatched_api_snippet                   1

Saved: c:\Users\grace\OneDrive\Bureau\McGill\COMP 370\COMP-370-Final-Project\data\cleaned\yoon_cleaned.csv
Saved: c:\Users\grace\OneDrive\Bureau\McGill\COMP 370\COMP-370-Final-Project\data\cleaned\yoon_excluded.csv


In [4]:
# Randomly select 50 for open coding. Preserve a previous sample if it exists.
sample_size = 50
sample_file = root / "data/annotated" / f"{annotator}_open_coding_50.csv"

if len(cleaned) < sample_size:
    raise ValueError(f"Only {len(cleaned)} eligible articles; need {sample_size} for open coding.")

if sample_file.exists():
    existing_ids = pd.read_csv(sample_file, usecols=["uuid"], dtype=str)["uuid"]
    if len(existing_ids) != sample_size or not set(existing_ids).issubset(set(cleaned["uuid"])):
        raise ValueError("Existing open-coding sample is incompatible with cleaned data. Back it up before changing it.")
    print(f"Preserved existing open-coding sample: {sample_file}")
else:
    sample = cleaned.sort_values("uuid").sample(n=sample_size, random_state=370)
    sample = sample[["uuid", "collector", "source", "published_at", "title", "opening", "url"]].copy()
    sample["topic_candidate"] = ""
    sample["notes"] = ""
    sample_file.parent.mkdir(parents=True, exist_ok=True)
    sample.to_csv(sample_file, index=False, encoding="utf-8-sig")
    print(f"Saved {sample_size} randomly selected articles: {sample_file}")

print("Open the sample CSV and fill topic_candidate / notes. No automatic labels are generated.")

Saved 50 randomly selected articles: c:\Users\grace\OneDrive\Bureau\McGill\COMP 370\COMP-370-Final-Project\data\annotated\yoon_open_coding_50.csv
Open the sample CSV and fill topic_candidate / notes. No automatic labels are generated.
